## Lab | First match ≠ right match

In [4]:
%pip install -U -q python-dotenv pypdf langchain-core langchain-community langchain-text-splitters langchain-openai pinecone langchain-pinecone

Note: you may need to restart the kernel to use updated packages.


In [5]:
import os
from dotenv import load_dotenv

load_dotenv()

for key in ["OPENAI_API_KEY", "PINECONE_API_KEY"]:
    if not os.getenv(key):
        raise ValueError(f"{key} not found. Add it to your .env file.")
print("API keys loaded")

API keys loaded


In [6]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, TextLoader

DATA_DIR = Path("data")
PODCAST_PATH = DATA_DIR / "podcast.txt"
PDF_PATH = DATA_DIR / "eu_ai_act.pdf"

for path in [PODCAST_PATH, PDF_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. Put the lab files in a 'data' folder next to this notebook.")

podcast_docs = TextLoader(str(PODCAST_PATH), encoding="utf-8").load()  # 1 Document (whole transcript)
pdf_docs = PyPDFLoader(str(PDF_PATH)).load()                           # 1 Document per PDF page

print(f"Podcast: {len(podcast_docs)} document, {len(podcast_docs[0].page_content):,} characters")
print(f"EU AI Act: {len(pdf_docs)} pages")
print("\nRaw metadata from the PDF loader:", pdf_docs[0].metadata)

/tmp/ipykernel_24431/320955083.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, TextLoader


Podcast: 1 document, 16,414 characters
EU AI Act: 144 pages

Raw metadata from the PDF loader: {'producer': 'PDFlib+PDI 9.0.7p3 (C++/Win64)', 'creator': 'Servigistics Arbortext Publishing Engine', 'creationdate': '2024-07-11T14:47:17+02:00', 'author': 'Publications Office of the European Union L-2985 Luxembourg LUXEMBOURG', 'keywords': 'ISSN 1977-0677', 'moddate': '2024-07-11T15:55:28+02:00', 'subject': 'I Legislative acts', 'title': 'Regulation (EU) 2024/1689 of the European Parliament and of the Council of 13 June 2024 laying down harmonised rules on artificial intelligence and amending Regulations (EC) No 300/2008, (EU) No 167/2013, (EU) No 168/2013, (EU) 2018/858, (EU) 2018/1139 and (EU) 2019/2144 and Directives 2014/90/EU, (EU) 2016/797 and (EU) 2020/1828 (Artificial Intelligence Act)Text with EEA relevance.', 'source': 'data/eu_ai_act.pdf', 'total_pages': 144, 'page': 0, 'page_label': '1'}


In [21]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import re

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

def chunk_with_metadata(docs, source_type, source_name):
    """Split documents into chunks and give every chunk a clean, consistent metadata schema."""
    chunks = splitter.split_documents(docs)
    
    track_sections = source_type == "regulation"
    section = "recital"   # the document starts with recitals
    article = None
    
    for i, chunk in enumerate(chunks):
        metadata = {
            "source_type": source_type,        # "podcast" or "regulation" -> used for filtering in Step 5
            "source_name": source_name,        # human-readable name -> used for citations
            "chunk_id": f"{source_type}-{i}",# unique id -> lets a reviewer trace an exact chunk
            
        }
        # PDF pages are 0-indexed in the loader; store the page number a human would see
        if "page" in chunk.metadata:
            metadata["page"] = int(chunk.metadata["page"]) + 1
            
        # new metadata - section 
        if track_sections:
            text = chunk.page_content

            # Section boundaries: the last marker in the chunk wins
            if re.search(r"^ANNEX I\s*$", text, flags=re.MULTILINE):
                section, article = "annex", None
            elif re.search(r"^CHAPTER I\s*$", text, flags=re.MULTILINE):
                section, article = "article", None

            # Article headings (alone on their line, so "Article 6(1)" in a sentence is ignored)
            if section == "article":
                headings = re.findall(r"^Article\s+(\d+)\s*$", text, flags=re.MULTILINE)
                if headings:
                    article = f"Article {headings[-1]}"

            metadata["section"] = section
            if article:                       # Pinecone rejects None, so omit the key
                metadata["article"] = article

        chunk.metadata = metadata
    return chunks

podcast_chunks = chunk_with_metadata(podcast_docs, "podcast", "Trustworthy AI podcast")
act_chunks = chunk_with_metadata(pdf_docs, "regulation", "EU AI Act (Reg. 2024/1689)")
all_chunks = podcast_chunks + act_chunks

print(f"Podcast chunks:   {len(podcast_chunks)}")
print(f"EU AI Act chunks: {len(act_chunks)}")
print(f"Total chunks:     {len(all_chunks)}")

Podcast chunks:   20
EU AI Act chunks: 764
Total chunks:     784


In [22]:
for chunk in [podcast_chunks[0], act_chunks[150]]:
    print(chunk.metadata)
    print(chunk.page_content[:300], "...")
    print("-" * 80)

{'source_type': 'podcast', 'source_name': 'Trustworthy AI podcast', 'chunk_id': 'podcast-0'}
So imagine for a second you're driving across a, I don't know, a massive suspension bridge. Okay. You don't pull over halfway across, get out and demand to see the blueprints, right? You don't interview the welding crew. No, you just, you trust it. You just drive, you trust the bridge, you trust the ...
--------------------------------------------------------------------------------
{'source_type': 'regulation', 'source_name': 'EU AI Act (Reg. 2024/1689)', 'chunk_id': 'regulation-150', 'page': 21, 'section': 'recital'}
indicators relating to AI.
OJ L, 12.7.2024 EN
ELI: http://data.europa.eu/eli/reg/2024/1689/oj 21/144
(35) Directive 2014/31/EU of the European Parliament and of the Council of 26 February 2014 on the harmonisation of the laws of the 
Member States relating to the making available on the market of no ...
-----------------------------------------------------------------------------

In [9]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector = embeddings.embed_query(str(all_chunks[0]))
print(len(vector))

1536


### Embeddings + Baseline Retrieval

In [23]:
from pinecone import Pinecone, ServerlessSpec
from uuid import uuid4

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
index_name = "lab-rerankers"   


# Start clean so old chunks without `section` don't linger
if pc.has_index(index_name):
    pc.delete_index(index_name)

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(index_name)

In [24]:
from langchain_pinecone import PineconeVectorStore

vector_store = PineconeVectorStore.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    index_name="my-index",
)

In [15]:
# PROVIDED: fixed test queries. Use the SAME queries in every step so you can compare fairly.
TEST_QUERIES = [
    "What AI practices are prohibited under the EU AI Act?",
    "What are the seven requirements for trustworthy AI?",
    "What is the difference between human-in-the-loop and human-on-the-loop?",
    "What fines apply for using a prohibited AI practice?",
    "Is social scoring of citizens allowed?",
]

#Helper to print results in the same format in every step
def show_results(query, results, label="Results"):
    """results: list of (Document, score) pairs"""
    print(f"\n=== {label} | Query: {query}")
    for rank, (doc, score) in enumerate(results, 1):
        m = doc.metadata
        where = f"p.{m['page']}" if "page" in m else m.get("chunk_id")
        print(f"#{rank}  score={score:.3f}  [{m.get('source_type')} | {where}]  {doc.page_content[:120]!r}")

In [16]:
def baseline_search(query, k=5):
    return vector_store.similarity_search_with_score(query, k=k)

baseline_results = {}
for q in TEST_QUERIES:
    baseline_results[q] = baseline_search(q)
    show_results(q, baseline_results[q])


=== Results | Query: What AI practices are prohibited under the EU AI Act?
#1  score=0.698  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'
#2  score=0.698  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'
#3  score=0.698  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'
#4  score=0.698  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'
#5  score=0.684  [regulation | p.12.0]  '(45) Practices that are prohibited by Union law, including data protection law, non-discrimination law, consumer \nprotec'

=== Results | Query: What are the seven requirements for trustworthy AI?
#1  score=0.674  [podcast 

### LLM Relevance Scoring

Goal: have an LLM judge ho relevant each retrieved chunk is, then re-order the results.

In [17]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


In [ ]:
baseline_results_2={}

for q in TEST_QUERIES:
    baseline_results_2[q] = baseline_search(q, k=10)
    show_results(q, baseline_results_2[q])



=== Results | Query: What AI practices are prohibited under the EU AI Act?
#1  score=0.698  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'
#2  score=0.698  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'
#3  score=0.698  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'
#4  score=0.684  [regulation | p.12.0]  '(45) Practices that are prohibited by Union law, including data protection law, non-discrimination law, consumer \nprotec'
#5  score=0.684  [regulation | p.12.0]  '(45) Practices that are prohibited by Union law, including data protection law, non-discrimination law, consumer \nprotec'
#6  score=0.684  [regulation | p.12.0]  '(45) Practices that are prohibited by Union law, including d

In [30]:
import re

JUDGE_PROMPT = """You are grading search results for relevance.
            Query: {query}
            Passage:{passage} 
            Rate how well the passage answers or is relevant to the query on a scale from 0.0 (irrelevant) to 1.0 (directly and fully answers it).
            Respond with only the number."""

llm_call_count = 0
def llm_relevance_score(query, doc):
    global llm_call_count
    prompt = JUDGE_PROMPT.format(query=query, passage=doc.page_content)
    try:
        llm_call_count+=1
        text = llm.invoke(prompt).content
        match = re.search(r"\d*\.?\d+", text)
        if not match:
            return 0.0
        return min(max(float(match.group()), 0.0), 1.0)  # clamp to [0, 1]
    except Exception:
        return 0.0  # don't let one failed call break the whole rerank
    
    

def llm_rerank(query, k_initial=10, k_final=5, w_sim=0.3, w_llm=0.7, candidates=None):
    # candidates: optional precomputed [(doc, sim_score), ...], e.g. baseline_results_2[query]
    if candidates is None:
        candidates = baseline_search(query, k=k_initial)

    scored = []
    for doc, sim in candidates[:k_initial]:
        llm_score = llm_relevance_score(query, doc)
        sim = min(max(sim, 0.0), 1.0)  # keep both scores on the same 0-1 scale
        combined = w_sim * sim + w_llm * llm_score
        scored.append((doc, combined))

    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:k_final]

In [ ]:
reranked_results = {}
llm_call_count = 0
for q in TEST_QUERIES:
    reranked_results[q] = llm_rerank(q, candidates=baseline_results_2[q])
    show_results(q, reranked_results[q])


=== Results | Query: What AI practices are prohibited under the EU AI Act?
#1  score=0.910  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'
#2  score=0.910  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'
#3  score=0.910  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'
#4  score=0.554  [regulation | p.9.0]  'intention to distort behaviour where the distortion results from factors external to the AI system which are outside \nth'
#5  score=0.554  [regulation | p.9.0]  'intention to distort behaviour where the distortion results from factors external to the AI system which are outside \nth'

=== Results | Query: What are the seven requirements for trustworthy AI?
#1  score=0.883  [regulation 

In [ ]:
print(f"LLM calls: {llm_call_count}")  # expect len(TEST_QUERIES) * 10

LLM calls: 50


### Dedicated Reranker (Cross-Encoder or Cohere)

### Metedata Filtering

In [18]:
#The filter is applied inside Pinecone before ranking, so you always get k results from that subset.

def filtered_search(query, filter, k=5):
    return vector_store.similarity_search_with_score(query, k=5, filter=filter)

query = TEST_QUERIES[0]

for source_type in ["regulation", "podcast"]:
    print(f"\n=== {source_type} only ===")
    for doc, score in filtered_search(query, {"source_type": {"$eq": source_type}}):
        print(f"{score:.4f}", doc.page_content[:150].replace("\n", " "))


=== regulation only ===
0.6985 Article 4 AI literacy Providers and deployers of AI systems shall take measures to ensure, to their best extent, a sufficient level of AI literacy of 
0.6985 Article 4 AI literacy Providers and deployers of AI systems shall take measures to ensure, to their best extent, a sufficient level of AI literacy of 
0.6984 Article 4 AI literacy Providers and deployers of AI systems shall take measures to ensure, to their best extent, a sufficient level of AI literacy of 
0.6984 Article 4 AI literacy Providers and deployers of AI systems shall take measures to ensure, to their best extent, a sufficient level of AI literacy of 
0.6840 (45) Practices that are prohibited by Union law, including data protection law, non-discrimination law, consumer  protection law, and competition law,

=== podcast only ===
0.5194 a pixel, it doesn't recognize or can be easily hacked, then it causes harm. An intentional harm is still harm. So lawful, ethical and robust, the docu
0.519

The regulation filter returned formal, definitional text, and the podcast filter returned conversational commentary that discusses the topic more loosely. 

In [19]:
#Before updating the metadata
filtered_search(query, {"section": {"$eq": "article"}})

[]

In [25]:
filtered_search(query, {"section": {"$eq": "article"}})

[(Document(id='c69f3cba-09bc-4ca4-a442-3416708b6b02', metadata={'article': 'Article 5', 'chunk_id': 'regulation-342', 'page': 51.0, 'section': 'article', 'source_name': 'EU AI Act (Reg. 2024/1689)', 'source_type': 'regulation'}, page_content='Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a sufficient level of AI literacy of \ntheir staff and other persons dealing with the operation and use of AI systems on their behalf, taking into account their \ntechnical knowledge, experience, education and training and the context the AI systems are to be used in, and considering \nthe persons or groups of persons on whom the AI systems are to be used.\nCHAPTER II\nPROHIBITED AI PRACTICES\nArticle 5\nProhibited AI practices\n1. The following AI practices shall be prohibited:\n(a) the placing on the market, the putting into service or the use of an AI system that deploys subliminal techniques beyond \na person’s consciousness or pu

In [26]:
# One specific article
filtered_search(query, {"article": {"$eq": "Article 5"}})

[(Document(id='c69f3cba-09bc-4ca4-a442-3416708b6b02', metadata={'article': 'Article 5', 'chunk_id': 'regulation-342', 'page': 51.0, 'section': 'article', 'source_name': 'EU AI Act (Reg. 2024/1689)', 'source_type': 'regulation'}, page_content='Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a sufficient level of AI literacy of \ntheir staff and other persons dealing with the operation and use of AI systems on their behalf, taking into account their \ntechnical knowledge, experience, education and training and the context the AI systems are to be used in, and considering \nthe persons or groups of persons on whom the AI systems are to be used.\nCHAPTER II\nPROHIBITED AI PRACTICES\nArticle 5\nProhibited AI practices\n1. The following AI practices shall be prohibited:\n(a) the placing on the market, the putting into service or the use of an AI system that deploys subliminal techniques beyond \na person’s consciousness or pu

In [27]:
filtered_search(query, {"source_type": {"$eq": "regulation"}, "section": {"$eq": "article"}})

[(Document(id='c69f3cba-09bc-4ca4-a442-3416708b6b02', metadata={'article': 'Article 5', 'chunk_id': 'regulation-342', 'page': 51.0, 'section': 'article', 'source_name': 'EU AI Act (Reg. 2024/1689)', 'source_type': 'regulation'}, page_content='Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a sufficient level of AI literacy of \ntheir staff and other persons dealing with the operation and use of AI systems on their behalf, taking into account their \ntechnical knowledge, experience, education and training and the context the AI systems are to be used in, and considering \nthe persons or groups of persons on whom the AI systems are to be used.\nCHAPTER II\nPROHIBITED AI PRACTICES\nArticle 5\nProhibited AI practices\n1. The following AI practices shall be prohibited:\n(a) the placing on the market, the putting into service or the use of an AI system that deploys subliminal techniques beyond \na person’s consciousness or pu

A filter can only narrow the search to the right region, and it can’t fix a wrong guess. If you filter on Article 5 for a question that’s really answered in Article 6, you’ll get plausible-looking but wrong results. That’s the trade-off to note for your write-up: filtering gives precision when you know where to look, and ranking has to do the work when you don’t.

**Note**: updating metadata without re-embedding

Pinecone stores each vector together with a snapshot of its metadata, so changing `chunk_with_metadata` in the notebook does not change what is already in the index. In this lab I deleted the index and re-uploaded everything, which is the simplest option but also re-embeds text that did not change.

Two ways to avoid that:

- **Deterministic IDs:** if each chunk has a stable ID (e.g. its `chunk_id`), re-running `add_documents` upserts over the old records and replaces their metadata, with no duplicates. This still re-embeds, since the upsert sends new vectors.
- **Metadata-only update:** `index.update(id=..., set_metadata={...})` patches the metadata on an existing vector and keeps its embedding. This requires knowing each vector's ID, so it only works if IDs were assigned deliberately (random `uuid4()` IDs make it awkward).

**Takeaway:** metadata has to be written at upsert or update time, so plan the schema (and use stable IDs) before the first upload. The only things that truly cannot change after index creation are the dimension and the metric.

### Complete RAG Pipeline with Reranking

In [28]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# Initialize LLM
# Reuses `llm` (gpt-4o-mini, temperature=0), `baseline_search` and `llm_rerank` from earlier


rag_prompt = ChatPromptTemplate.from_template(
    """Answer the question using ONLY the context below.

Rules:
- After each claim, cite the source label exactly as it appears in the context, e.g. [EU AI Act, p.51].
- If the context does not contain the answer, reply exactly: "I don't know based on the sources."
- Do not use outside knowledge.

Context:
{context}

Question: {question}

Answer:"""
)


#Retrieval methods: each takes a query string and returns a list of Documents
def retrieve_baseline(query, k=5):
    return [doc for doc, _ in baseline_search(query, k=k)]

def retrieve_reranked(query, k=5):
    return [doc for doc, _ in llm_rerank(query, k_initial=10, k_final=k)]


# Helper to format retrieved documents into a single string
def format_docs(docs):
    blocks = []
    for doc in docs:
        m = doc.metadata
        label = m.get("source_name", "unknown source")
        if "page" in m:                       # the podcast has no page numbers
            label += f", p.{m['page']}"
        blocks.append(f"[{label}]\n{doc.page_content}")
    return "\n\n".join(blocks)


def build_rag_chain(retrieve_fn):
    return (
        {
            "context": RunnableLambda(retrieve_fn) |format_docs,
            "question": RunnablePassthrough(),
        }
        | rag_prompt
        | llm
        | StrOutputParser()
    )

rag_baseline = build_rag_chain(retrieve_baseline)
rag_reranked = build_rag_chain(retrieve_reranked)


In [31]:
question = "What counts as a high-risk AI system?"

for name, chain in [("baseline", rag_baseline), ("reranked", rag_reranked)]:
    print(f"--- {name} ---")
    print(chain.invoke(question), "\n")

# Test the "I don't know" behavior with something the sources shouldn't cover
print(rag_baseline.invoke("What is the best pizza topping?"))

--- baseline ---
High-risk AI systems are those listed in the following areas: 
1. Biometrics, including:
   (a) remote biometric identification systems (excluding systems for biometric verification to confirm identity);
   (b) AI systems for biometric categorisation based on sensitive or protected attributes;
   (c) AI systems for emotion recognition.
2. Critical infrastructure, specifically AI systems used as safety components in managing and operating critical digital infrastructure, road traffic, or in supplying water, gas, heating, or electricity.
3. Education and vocational training. [EU AI Act (Reg. 2024/1689), p.127.0] 

--- reranked ---
High-risk AI systems are those listed in the following areas: 

1. Biometrics, where their use is permitted under relevant Union or national law, including:
   (a) remote biometric identification systems (excluding those for biometric verification to confirm identity);
   (b) AI systems for biometric categorisation based on sensitive or protect

### Evaluate: With vs. Without Reranking

In [32]:
import time

def answer(q, retrieve_fn):
    """Run the RAG chain with the given retrieval function. Returns (answer_text, seconds)."""
    chain = build_rag_chain(retrieve_fn)
    start = time.perf_counter()
    text = chain.invoke(q)
    return text, time.perf_counter() - start

for q in TEST_QUERIES:
    base_ans, base_t = answer(q, retrieve_baseline)
    rerank_ans, rerank_t = answer(q, retrieve_reranked)

    print("=" * 80)
    print(f"Q: {q}\n")
    print(f"BASELINE ({base_t:.2f}s):\n{base_ans}\n")
    print(f"RERANKED ({rerank_t:.2f}s):\n{rerank_ans}\n")

Q: What AI practices are prohibited under the EU AI Act?

BASELINE (4.34s):
The following AI practices shall be prohibited under the EU AI Act: (a) the placing on the market, the putting into service or the use of an AI system that deploys subliminal techniques beyond a person’s consciousness or purposefully manipulative or deceptive techniques, with the objective, or the effect of materially distorting the behaviour of a person or a group of persons by appreciably impairing their ability to make an [EU AI Act (Reg. 2024/1689), p.51.0].

RERANKED (11.26s):
The following AI practices shall be prohibited under the EU AI Act: (a) the placing on the market, the putting into service or the use of an AI system that deploys subliminal techniques beyond a person’s consciousness or purposefully manipulative or deceptive techniques, with the objective, or the effect of materially distorting the behaviour of a person or a group of persons by appreciably impairing their ability to make an [EU AI A

| Query | Method | Top-1 chunk relevant? (Y/N) | Answer correct? (Y/Partial/N) | Grounded in cited chunk? (Y/N) | Notes |
|---|---|---|---|---|---|
| Q1 | Baseline | Y | Y | Y | 10 |
| Q1 | Reranked | Y | Y | Y | 10 |
| Q2 | Baseline | N | N | Y | 1 |
| Q2 | Reranked | Y | Y | Y | 8 |
| Q3 | Baseline | Y | Y | Y | 10 |
| Q3 | Reranked | Y | Y | Y | 10 |
| Q4 | Baseline | - | Y | - | 10 |
| Q4 | Reranked | - | Y | - | 10 |
| Q5 | Baseline | Y | Y | Y | 8 |
| Q5 | Reranked | Y | Y | Y | 8 |

Reranking was better at retrieving an answer for Q2 whereas Baseline method wasn't able to retrieve one and answered with "I don't know based on the sources."

In [34]:
import time
import pandas as pd
from langchain_community.callbacks import get_openai_callback

def measure(q, retrieve_fn):
    chain = build_rag_chain(retrieve_fn)   # original chain: returns just the answer string
    with get_openai_callback() as cb:
        start = time.perf_counter()
        chain.invoke(q)
        secs = time.perf_counter() - start
    return {
        "seconds": secs,
        "llm_calls": cb.successful_requests,
        "tokens": cb.total_tokens,
        "cost_usd": cb.total_cost,
    }

rows = []
for q in TEST_QUERIES:
    for name, fn in [("baseline", retrieve_baseline), ("reranked", retrieve_reranked)]:
        rows.append({"query": q, "method": name, **measure(q, fn)})

df = pd.DataFrame(rows)

summary = df.groupby("method")[["seconds", "llm_calls", "tokens", "cost_usd"]].mean()
print(summary.round(4))

b, r = summary.loc["baseline"], summary.loc["reranked"]
print(f"\nExtra LLM calls per query: {r.llm_calls - b.llm_calls:.0f}")
print(f"Extra seconds per query:   {r.seconds - b.seconds:.2f}  ({r.seconds / b.seconds:.1f}x slower)")
print(f"Extra tokens per query:    {r.tokens - b.tokens:.0f}")
print(f"Extra cost per query:      ${r.cost_usd - b.cost_usd:.5f}  ({r.cost_usd / b.cost_usd:.1f}x)")
print(f"Extra cost per 1,000 queries: ${(r.cost_usd - b.cost_usd) * 1000:.2f}")

          seconds  llm_calls  tokens  cost_usd
method                                        
baseline   2.3564        1.0  1215.8    0.0002
reranked  10.1561       11.0  3926.2    0.0006

Extra LLM calls per query: 10
Extra seconds per query:   7.80  (4.3x slower)
Extra tokens per query:    2710
Extra cost per query:      $0.00041  (3.4x)
Extra cost per 1,000 queries: $0.41


**Quality gain:** on q2, the baseline answered "I don't know based on the sources", while the reranked chain found and cited an answer. The baseline only sees the top 5 by embedding similarity, so the relevant chunk was probably ranked 6-10 and the LLM judge promoted it into the top 5.

**Would a client pay for that?** Money is not the obstacle, since +$0.41 per 1,000 queries is negligible. Latency is the real cost: about 10 seconds per answer is too slow for a live chat widget but acceptable for research, compliance or legal review, where a missed answer or wrong citation is costly.

**Ways to cut the overhead:** run the judge calls in parallel (`llm.batch`), lower `k_initial`, use a cross-encoder reranker instead of an LLM judge, or rerank only when the baseline's top similarity score is low.

**Conclusion:** worth it for high-stakes regulatory Q&A. Not worth it for casual or latency-sensitive lookups unless the judge calls are parallelized.